# Chirp AI — Detector de Toxicidad
### Proyecto Final · Inteligencia Artificial y MLOps (UCB)

Modelo de **clasificación de texto (NLP)** que detecta si un chirp es **tóxico/ofensivo** y resalta las palabras problemáticas.

**Pipeline**: TF-IDF (unigramas) + Regresión Logística, comparado con otros clasificadores. Se exporta como `.pkl` que consume el microservicio FastAPI.

> **Dataset recomendado:** *Jigsaw Toxic Comment Classification* de Kaggle (columna `comment_text` + etiqueta `toxic`). Descargar el CSV a `notebook/data/` y renombrarlo a `toxicity.csv`. Si no está, el notebook genera un dataset sintético balanceado para ejecutarse igual.

## 0. Instalación (Colab)

In [ ]:
# !pip install scikit-learn pandas numpy matplotlib seaborn joblib -q
import warnings; warnings.filterwarnings('ignore')

## 1. Imports y utilidades compartidas

Reutilizamos `moderation.py` del microservicio (misma tokenización y pesos por palabra en entrenamiento e inferencia).

In [ ]:
import os, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sys.path.append(os.path.abspath('../ai-service'))
from app.moderation import make_synthetic_dataset, word_weights

## 2. Carga de datos

In [ ]:
DATA_PATH = 'data/toxicity.csv'

def load_real(path):
    df = pd.read_csv(path)
    rename = {'comment_text':'text','tweet':'text','content':'text',
              'Toxic':'toxic','label':'toxic','class':'toxic'}
    df = df.rename(columns={k:v for k,v in rename.items() if k in df.columns})
    df = df[['text','toxic']].dropna()
    df['toxic'] = (df['toxic'] > 0).astype(int)  # binariza por si viene 0..1 o multi
    return df

if os.path.exists(DATA_PATH):
    df = load_real(DATA_PATH); print('Dataset real:', df.shape)
else:
    texts, labels = make_synthetic_dataset(6000)
    df = pd.DataFrame({'text': texts, 'toxic': labels})
    print('CSV no encontrado -> dataset SINTETICO:', df.shape)
df.head()

## 3. Control de calidad de datos

In [ ]:
print('Nulos:\n', df.isna().sum())
print('Duplicados:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df = df[df['text'].astype(str).str.strip() != ''].reset_index(drop=True)
print('Balance de clases:\n', df['toxic'].value_counts())

## 4. EDA

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
df['toxic'].value_counts().plot(kind='bar', ax=ax[0], color=['#1d9bf0','#e0245e'])
ax[0].set_title('Distribucion de clases (0=ok, 1=toxico)')
df['len'] = df['text'].astype(str).str.len()
sns.boxplot(data=df, x='toxic', y='len', ax=ax[1])
ax[1].set_title('Longitud del texto por clase'); plt.tight_layout(); plt.show()

## 5. Vectorización y split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
X_train, X_test, y_train, y_test = train_test_split(
    df['text'].astype(str), df['toxic'], test_size=0.2, random_state=42, stratify=df['toxic'])
vec = TfidfVectorizer(ngram_range=(1,1), min_df=2)
Xtr = vec.fit_transform(X_train); Xte = vec.transform(X_test)
print('Vocabulario:', len(vec.vocabulary_))

## 6. Comparación de modelos

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
models = {'LogisticRegression': LogisticRegression(max_iter=1000, C=4.0),
          'LinearSVC': LinearSVC(),
          'MultinomialNB': MultinomialNB()}
res = {}
for name, m in models.items():
    m.fit(Xtr, y_train); p = m.predict(Xte)
    res[name] = {'accuracy':accuracy_score(y_test,p),'precision':precision_score(y_test,p),
                 'recall':recall_score(y_test,p),'f1':f1_score(y_test,p)}
res_df = pd.DataFrame(res).T.sort_values('f1', ascending=False); res_df

## 7. Evaluación del mejor modelo (matriz de confusión + reporte)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
best = LogisticRegression(max_iter=1000, C=4.0).fit(Xtr, y_train)
pred = best.predict(Xte)
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(4,3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['ok','toxico'], yticklabels=['ok','toxico'])
plt.title('Matriz de confusion'); plt.ylabel('real'); plt.xlabel('predicho'); plt.show()
print(classification_report(y_test, pred, target_names=['ok','toxico']))

## 8. Interpretación: palabras más tóxicas

Los coeficientes del modelo indican qué palabras empujan hacia 'tóxico'. Son las mismas que el microservicio usa para resaltar.

In [ ]:
import numpy as np
vocab = np.array(vec.get_feature_names_out())
coef = best.coef_[0]
top_tox = vocab[np.argsort(coef)[-20:]][::-1]
top_ok  = vocab[np.argsort(coef)[:15]]
print('Top palabras TOXICAS:', list(top_tox))
print('\nTop palabras NO toxicas:', list(top_ok))
plt.figure(figsize=(8,5))
vals = np.sort(coef)[-20:]
plt.barh(range(20), vals, color='#e0245e'); plt.yticks(range(20), vocab[np.argsort(coef)[-20:]])
plt.title('Palabras con mayor peso hacia toxico'); plt.tight_layout(); plt.show()

## 9. Exportar el Pipeline para el microservicio

In [ ]:
import joblib
from sklearn.pipeline import Pipeline
# Pipeline final (vectorizador + modelo) reentrenado con TODOS los datos.
final = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,1), min_df=2)),
                  ('clf', LogisticRegression(max_iter=1000, C=4.0))])
final.fit(df['text'].astype(str), df['toxic'])
os.makedirs('../ai-service/models', exist_ok=True)
joblib.dump(final, '../ai-service/models/toxicity_model.pkl')
print('Modelo guardado en ../ai-service/models/toxicity_model.pkl')

## 10. Demo de inferencia + resaltado

In [ ]:
for t in ['gracias por el gran trabajo equipo', 'eres un idiota, callate']:
    proba = final.predict_proba([t])[0][1]
    tokens = word_weights(final, t)
    flagged = [w['text'] for w in tokens if w['weight']>0]
    print(f'toxico={proba:.0%} | resaltadas={flagged} | {t}')

## 11. Conclusiones

- Modelo de NLP reproducible para moderación de contenido.
- La misma lógica (`moderation.py`) se usa al entrenar y al servir.
- El `.pkl` se despliega vía FastAPI y lo consume el backend NestJS, cerrando el ciclo de MLOps (entrenar → versionar → servir → consumir).
- Trabajo futuro: manejar sarcasmo/ofuscación, multilingüe, reentrenamiento con reportes de usuarios y monitoreo de drift.